# BCPC surface: arc_length_parallel and arc_length_orthogonal from the saved BCPC bundle

This notebook replaces `notebooks/arc_length_surface.ipynb` and `scripts/stakes_surface_pipeline.py`, which fitted their own BCPC. It fits nothing new on the BCPC side. For each model it reads the full-fit BCPC bundle that `notebooks/bcpc_out_of_fold.ipynb` saved in `artifacts/<model>/bcpc/` and takes two things from it: the target (`bcpc_arc_length`) and the stakes classes (nine, with `near_existential` merged into `existential`). It fits exactly the bundle's rows, so the two templates the bundle leaves out, `conversational_no_time--brief` and `conversational_no_time--notes_heading` (`cv.EXCLUDED_TEMPLATES`), are left out here too. On that basis it fits the same surface geometry, with the same weighting, as before:

1. 16 centred, unscaled PLS components against `bcpc_arc_length`
2. per-template class-mean splines in PLS1-3, in ladder order, and PLS1-3 rotated into the splines' best-fit shared plane
3. the cubic graph PLS2 = f(PLS1, PLS3), with its origin at the `very_low` PLS1 centroid
4. 5,000 fixed-PLS3 slices, validated on a sample, onto which every row is mapped

`arc_length_parallel` is the signed length along a row's slice, measured from the `very_low` origin. `arc_length_orthogonal` is the snapped PLS3 height, which is a legacy name, not a geodesic length. `refined_stakes` is an affine rescaling of `bcpc_arc_length`, so a PLS fit against either finds the same directions. The weighting is the old pipeline's. PLS gives every template file equal total weight. The centroid-plane rotation and the `very_low` origin count every template equally. The surface fit counts every row equally. These are in-sample descriptive fits, not held-out evaluations.

Everything runs on the CPU from cached activations. Outputs go beside each model's BCPC bundle:

- `bcpc/surface/`: `model.npz`, `model.json`, `rows.parquet`, and `plots/*.html`
- `bcpc/inference/<dataset>.csv`: every cached inference corpus mapped through the surface, with the bundle's `bcpc_arc_length` and `refined_stakes`. The original `inference/<dataset>/` CSVs are left untouched.

In [ ]:
from pathlib import Path
import gc
import sys
import traceback

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'bcpc_surface.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import bcpc_surface as bs
from scripts import inference_datasets
from scripts.pipeline_config import discover_run_dirs

print('Repository:', ROOT)
print(f'{len(inference_datasets.DATASETS)} inference corpora:',
      ', '.join(dataset.name for dataset in inference_datasets.DATASETS))

## Configuration

`MODELS = None` analyses every model directory under `ARTIFACT_ROOT` that has a BCPC bundle (`bcpc/rows.parquet`). Models without one are reported and skipped; at present that is only Qwen3-4B. A list of directory names restricts the run to those models, and each of them must have a bundle.

`SETTINGS` holds the surface settings, whose defaults match the old pipeline: 16 PLS components, cubic template splines with one interior knot, and 5,000 slices at 1e-4 relative arc tolerance. `PLOT_ROWS` only thins the 3D scatters for display; every number uses every row.

In [ ]:
ARTIFACT_ROOT = ROOT / 'artifacts' / 'content' / 'artifacts'
MODELS = None                   # None = every model with a BCPC bundle
SETTINGS = bs.SurfaceSettings()
PLOT_ROWS = 8_000               # rows drawn per 3D plot; None draws all

## Analysis helper

`analyse_model` handles one model from start to finish: fit, save, plot, then project every cached inference corpus. Before fitting, it checks that the bundle reproduces its saved arc lengths from the activations, so the target always matches these caches.

It shows only the plots and a few progress lines. The fit diagnostics (PLS variance and R², plane rotation, surface fit and slice validation) are written to `bcpc/surface/model.json`.

In [ ]:
def analyse_model(run_dir):
    result = bs.fit_surface(run_dir, SETTINGS)
    config = result.config
    print()
    print(f'=== {config.model_name}: {config.layer_component} ===')
    print(f'{len(result.rows):,} rows; bundle arc length reproduced to '
          f'{result.diagnostics["bundle_arc_length_max_error"]:.3g}')
    output_dir = bs.save(result, notebook='notebooks/pls_arc_length_surface.ipynb')
    print('Saved surface to', output_dir)
    for figure in bs.save_plots(result, PLOT_ROWS).values():
        figure.show()
    del result
    gc.collect()

    for name, (csv_path, _, _) in bs.project_inference(config).items():
        print(f'{inference_datasets.by_name(name).label} CSV:', csv_path)
    skipped = [d.name for d in inference_datasets.DATASETS
               if d not in inference_datasets.cached_datasets(config)]
    if skipped:
        print('No cached activations yet for:', ', '.join(skipped))
    return output_dir

## Analyse every model with a BCPC bundle

One model failing does not stop the others. Its traceback is printed, and once every model has run the cell raises.

In [ ]:
run_dirs = discover_run_dirs(ARTIFACT_ROOT)
if MODELS is not None:
    missing = [name for name in MODELS
               if ARTIFACT_ROOT / name not in run_dirs or not bs.has_bcpc_bundle(ARTIFACT_ROOT / name)]
    if missing:
        raise ValueError(f'No cached run with a BCPC bundle for {missing} under {ARTIFACT_ROOT}.')
    run_dirs = [ARTIFACT_ROOT / name for name in MODELS]
skipped = [path.name for path in run_dirs if not bs.has_bcpc_bundle(path)]
run_dirs = [path for path in run_dirs if bs.has_bcpc_bundle(path)]
if skipped:
    print('No BCPC bundle, skipped:', ', '.join(skipped))
if not run_dirs:
    raise ValueError(f'No model under {ARTIFACT_ROOT} has a BCPC bundle; '
                     'run notebooks/bcpc_out_of_fold.ipynb first.')
print(f'{len(run_dirs)} model(s):', ', '.join(path.name for path in run_dirs))

run_results = {}
for run_dir in run_dirs:
    try:
        output_dir = analyse_model(run_dir)
        run_results[run_dir.name] = {'status': 'complete', 'output_dir': str(output_dir)}
    except Exception as exc:
        run_results[run_dir.name] = {'status': 'failed', 'error': f'{type(exc).__name__}: {exc}',
                                     'traceback': traceback.format_exc()}
        print(run_results[run_dir.name]['traceback'])
    gc.collect()

failures = [name for name, result in run_results.items() if result['status'] != 'complete']
if failures:
    raise RuntimeError(f'Analysis failures: {failures}')

## Reusing a saved surface

```python
from scripts import bcpc_bundle, bcpc_surface
from scripts.pipeline_config import load_run_config

config = load_run_config(ARTIFACT_ROOT / 'Qwen3-32B')
arrays, metadata, coordinates = bcpc_surface.load_surface(config)   # refuses a changed BCPC bundle
pls_scores = (activation_batch - arrays['pls_mean']) @ arrays['pls_rotations']
surface_coordinates = coordinates.map_points(pls_scores[:, :3])      # arc_length_parallel, arc_length_orthogonal, ...
fit, _ = bcpc_bundle.load_bcpc_bundle(config.bcpc_dir)
bcpc_scores = bcpc_bundle.score_activations(fit, activation_batch)   # bcpc_arc_length, refined_stakes
```